# # Film Eleştirileri ve Bag-of-Words Modellemesi

🎯 Bu zorluğun amacı, metinlerin ***Bag-of-words*** modellemesiyle oynamaktır.

✍️ Aşağıdaki veri setinde, _“olumlu”_ veya _“olumsuz”_ olarak sınıflandırılmış 2000 adet yorum bulunmaktadır.

In [1]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/movie_reviews.csv")
data.head()

,target,reviews
0,neg,"plot : two teen couples go to a church party ,..."
1,neg,the happy bastard's quick movie review \ndamn ...
2,neg,it is movies like these that make a jaded movi...
3,neg,""" quest for camelot "" is warner bros . ' firs..."
4,neg,synopsis : a mentally unstable man undergoing ...


In [2]:
data.shape

(2000, 2)

## 1. Ön işleme (Preprocessing)

❓ **Soru (Metin Temizleme)** ❓

- Bir cümleyi temizleyecek bir `preprocessing` fonksiyonu yazın ve bunu tüm yorumlara uygulayın. Fonksiyon şunları yapmalıdır:
    - boşlukları kaldırma
    - harfleri küçük harfe çevirme
    - sayıları kaldırma
    - noktalama işaretlerini kaldırma
    - tokenization (kelimelere ayırma)
    - lemmatization (kelime köküne indirgeme)
- Temizlenmiş yorumları `clean_reviews` adlı bir sütunda saklayabilirsiniz.
- Bu aşamada stopword’leri kaldırmayın; nedenini `3. N-gram modelleme` bölümünde açıklayacağız.

In [3]:
import string
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def preprocessing(sentence):
    # 1) boşlukları kırp + küçük harf
    sentence = sentence.strip().lower()
    # 2) sayıları kaldır
    sentence = "".join(k for k in sentence if not k.isdigit())
    # 3) noktalama işaretlerini kaldır
    sentence = sentence.translate(str.maketrans("", "", string.punctuation))
    # 4) kelimelere ayır
    kelimeler = word_tokenize(sentence)
    # 5) lemmatize (önce fiil, sonra isim)
    kelimeler = [lemmatizer.lemmatize(lemmatizer.lemmatize(k, pos="v"), pos="n") for k in kelimeler]
    return " ".join(kelimeler)

preprocessing("I didn't LIKE the 2 movies, they were boring!")


'i didnt like the movie they be bore'

In [4]:
# Yorumları temizle
data["clean_reviews"] = data["reviews"].apply(preprocessing)
data[["reviews", "clean_reviews"]].head()

,reviews,clean_reviews
0,"plot : two teen couples go to a church party ,...",plot two teen couple go to a church party drin...
1,the happy bastard's quick movie review \ndamn ...,the happy bastard quick movie review damn that...
2,it is movies like these that make a jaded movi...,it be movie like these that make a jade movie ...
3,""" quest for camelot "" is warner bros . ' firs...",quest for camelot be warner bros first feature...
4,synopsis : a mentally unstable man undergoing ...,synopsis a mentally unstable man undergo psych...


❓ **Soru (LabelEncoding)**❓

Hedefinizi LabelEncode ile kodlayın ve `“target_encoded”` adlı bir sütuna kaydedin.

In [5]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
data["target_encoded"] = encoder.fit_transform(data["target"])

print(dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))
data["target_encoded"].value_counts()

{'neg': 0, 'pos': 1}


target_encoded
0    1000
1    1000
Name: count, dtype: int64

In [6]:
# Hızlı kontrol
data.head()

,target,reviews,clean_reviews,target_encoded
0,neg,"plot : two teen couples go to a church party ,...",plot two teen couple go to a church party drin...,0
1,neg,the happy bastard's quick movie review \ndamn ...,the happy bastard quick movie review damn that...,0
2,neg,it is movies like these that make a jaded movi...,it be movie like these that make a jade movie ...,0
3,neg,""" quest for camelot "" is warner bros . ' firs...",quest for camelot be warner bros first feature...,0
4,neg,synopsis : a mentally unstable man undergoing ...,synopsis a mentally unstable man undergo psych...,0


## 2. Bag-of-Words Modellemesi

❓ **Soru (Tek kelimelik sözcüklerle NaiveBayes)** ❓

`cross_validate` kullanarak, metinlerin Bag-of-Words temsilinde eğitilmiş bir Multinomial Naive Bayes modelini puanlayın.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_validate

vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(data["clean_reviews"])
print("Şekil:", X_bow.shape)

naivebayes = MultinomialNB()
cv_nb = cross_validate(naivebayes, X_bow, data["target_encoded"], scoring="accuracy")

print("Unigram BoW accuracy:", round(cv_nb["test_score"].mean(), 4))

Şekil: (2000, 36423)
Unigram BoW accuracy: 0.817


## 3. N-gram Modellemesi

👀 Stop kelimeleri kaldırmamanızı istediğimizi hatırlayın. Neden? 

👉 Naive Bayes modelini bigramlarla eğiteceğiz. Bu nedenle, “I do not like coriander” (Kişnişi sevmiyorum) gibi bir cümlede, örneğin bu cümlede olumsuzluğu tespit etmek için “do not” bigramını taramak önemlidir.

❓ **Soru (bigramlarla NaiveBayes)** ❓

`cross_validate` kullanarak, metinlerin 2-gram Bag-of-Words temsilinde eğitilmiş bir Multinomial Naive Bayes modelini puanlayın.

In [8]:
vectorizer = CountVectorizer(ngram_range = (2,2))
naivebayes = MultinomialNB()

X_bow = vectorizer.fit_transform(data.clean_reviews)

cv_nb = cross_validate(
    naivebayes,
    X_bow,
    data.target_encoded,
    scoring = "accuracy"
)

round(cv_nb['test_score'].mean(),2)

0.84

In [9]:
for aralik in [(1, 1), (2, 2), (1, 2)]:
    X = CountVectorizer(ngram_range=aralik).fit_transform(data["clean_reviews"])
    skor = cross_validate(MultinomialNB(), X, data["target_encoded"], scoring="accuracy")["test_score"]
    print(f"ngram_range={aralik}  kolon={X.shape[1]:>7,}  accuracy = {skor.mean():.4f} ± {skor.std():.4f}")

ngram_range=(1, 1)  kolon= 36,423  accuracy = 0.8170 ± 0.0130
ngram_range=(2, 2)  kolon=447,377  accuracy = 0.8415 ± 0.0179
ngram_range=(1, 2)  kolon=483,800  accuracy = 0.8305 ± 0.0086


🏁 Tebrikler! Artık vektörleştirilmiş metinler üzerinde Naive Bayes modelini nasıl eğiteceğinizi biliyorsunuz.

💾 Not defterinizi `git add/commit/push` yapmayı unutmayın...

